In [17]:
import os
os.chdir(r"C:\vscode\graph-rag\Source")

In [18]:
from config.settings_loader import load_config

config = load_config("config/config.yaml")

In [4]:
import pdfplumber

text = []
with pdfplumber.open(config["data_source"]["raw_data"]["pdf_path"]) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text.append(page_text)

raw_text = "\n\n".join(text)

In [ ]:
with open(config["data_source"]["normalized_data"]["unstructured_text_path"], "w", encoding="utf-8") as f:
    f.write(raw_text)

In [ ]:
#to add ## and # for chapters and parts
import re

def add_md_headings(text: str) -> str:
    lines = text.splitlines()
    out = []

    for line in lines:
        stripped = line.strip()

        # Chapter heading
        if re.match(r"^Chapter\s+[IVXLC]+:", stripped):
            out.append(f"## {stripped}")
            continue

        # Part heading (—Part I., —Part II., etc.)
        if re.search(r"—Part\s+[IVXLC]+\.?", stripped):
            out.append(f"### {stripped}")
            continue

        out.append(line)

    return "\n".join(out)

#to remove the asterisks from the text
import re

def remove_unnecessary_asterisks(text: str) -> str:
    # Remove "* Note:" or "* note:" (editorial markers)
    text = re.sub(r"\*\s*note\s*:", "", text, flags=re.IGNORECASE)

    # Remove inline sequences of 2+ asterisks (spaced or unspaced)
    text = re.sub(r"(\*\s*){2,}", "", text)

    cleaned_lines = []
    for line in text.splitlines():
        stripped = line.strip()

        # Remove lines containing only asterisks (any spacing)
        if re.fullmatch(r"(\*\s*)+", stripped):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)

def fix_line_wraps(text: str) -> str:
    lines = text.splitlines()
    out = []
    buffer = ""

    for line in lines:
        stripped = line.rstrip()

        # Preserve empty lines (paragraph boundaries)
        if not stripped:
            if buffer:
                out.append(buffer.strip())
                buffer = ""
            out.append("")
            continue

        # Preserve markdown headings
        if stripped.startswith("#"):
            if buffer:
                out.append(buffer.strip())
                buffer = ""
            out.append(stripped)
            continue

        # Preserve footnote markers & blocks for now
        if stripped.startswith("[") or stripped.endswith("]") or "(return)" in stripped:
            if buffer:
                out.append(buffer.strip())
                buffer = ""
            out.append(stripped)
            continue

        # Join wrapped lines
        if buffer:
            buffer += " " + stripped
        else:
            buffer = stripped

    if buffer:
        out.append(buffer.strip())

    return "\n".join(out)

#to remove square bracket footnotes
def remove_square_bracket_footnotes(text: str) -> str:
    """
    Removes all content enclosed in square brackets [ ... ],
    including multi-line scholarly footnotes.
    """
    # Remove multi-line square bracket blocks
    text = re.sub(r"\[\s*.*?\s*\]", "", text, flags=re.DOTALL)

    # Clean up excessive blank lines left behind
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


In [35]:
# read normalized text
with open(
    config["data_source"]["normalized_data"]["unstructured_text_path"],
    "r",
    encoding="utf-8"
) as f:
    data = f.read()

# add markdown headings
structured_data = add_md_headings(data)
structured_data = remove_unnecessary_asterisks(structured_data)
structured_data = fix_line_wraps(structured_data)

# write to structured location
with open(
    config["data_source"]["normalized_data"]["structured_text_path"],
    "w",
    encoding="utf-8"
) as f:
    f.write(structured_data)

In [ ]:
# Over-aggressive paragraph merging

# Paragraph boundaries destroyed

# Footnotes leaking into narrative text

# Inline citation markers (1a, 5, etc.) not removed

# Editorial notes (—M.], —G.]) present

# Unrelated sections merged together

# Paragraphs exceeding reasonable length

# Structural markers not cleanly isolated

# Sentence order preserved but semantic flow corrupted